In [ ]:

!wget -q https://github.com/jbrownlee/Datasets/releases/download/Flickr8k/Flickr8k_Dataset.zip
print("Unzipping images...")
!unzip -q Flickr8k_Dataset.zip -d /content/Flickr8k_Dataset
!rm Flickr8k_Dataset.zip
print("Done! Check your sidebar now.")

Unzipping images...
Done! Check your sidebar now.


In [ ]:
import os
import string
import numpy as np
import pickle
import random
import tensorflow as tf
from google.colab import drive
from keras.models import Model
from keras.layers import Layer, Input, Dense, LSTM, Embedding, Dropout, concatenate
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from keras.applications.xception import Xception
from PIL import Image

drive.mount('/content/drive')
DRIVE_PATH = "/content/drive/MyDrive/Image_Captioning"
dataset_text = os.path.join(DRIVE_PATH, "Flickr8k_text")

#unzipped files in local session storage
dataset_images = "/content/Flickr8k_Dataset/Flicker8k_Dataset"

features_file = os.path.join(DRIVE_PATH, "features_attention.p")
tokenizer_file = os.path.join(DRIVE_PATH, "tokenizer.p")
descriptions_file = os.path.join(DRIVE_PATH, "descriptions.txt")
models_dir = os.path.join(DRIVE_PATH, "models_attention")

if not os.path.exists(models_dir):
    os.makedirs(models_dir)

if not os.path.exists(features_file):
    print("Features file not found! Extracting raw spatial grid features (10x10x2048)...")

    xception_spatial = Xception(include_top=False, weights='imagenet')

    train_imgs_file = os.path.join(dataset_text, "Flickr_8k.trainImages.txt")
    with open(train_imgs_file, 'r') as f:
        train_photos = f.read().split("\n")[:-1]

    features = {}
    print(f"Extracting features for {len(train_photos)} training images...")
    for img in train_photos:
        img_path = os.path.join(dataset_images, img)
        if os.path.exists(img_path):

            image = Image.open(img_path).convert('RGB')
            image = image.resize((299, 299))

            image = np.array(image)
            image = np.expand_dims(image, axis=0) / 127.5 - 1.0

            spatial_feat = xception_spatial.predict(image, verbose=0)

            features[img] = np.reshape(spatial_feat, (100, 2048))

    with open(features_file, "wb") as f:
        pickle.dump(features, f)
    print("Spatial grid features extracted and saved successfully!")
else:
    print("Spatial features file found! Loading existing array maps...")
    with open(features_file, "rb") as f:
        train_features = pickle.load(f)

def load_doc(filename):
    with open(filename, 'r') as file: return file.read()

train_imgs_file = os.path.join(dataset_text, "Flickr_8k.trainImages.txt")
train_imgs = load_doc(train_imgs_file).split("\n")[:-1]

with open(tokenizer_file, 'rb') as f:
    tokenizer = pickle.load(f)
vocab_size = len(tokenizer.word_index) + 1
max_length = 32

with open(features_file, "rb") as f:
    train_features = pickle.load(f)

# Ensure data alignment
train_imgs = [img for img in train_imgs if img in train_features]

def load_clean_descriptions(filename, photos):
    file = load_doc(filename)
    descriptions = {}
    for line in file.split("\n"):
        words = line.split()
        if len(words) < 1: continue
        image, image_caption = words[0], words[1:]
        if image in photos:
            if image not in descriptions:
                descriptions[image] = []
            descriptions[image].append('<start> ' + " ".join(image_caption) + ' <end>')
    return descriptions

train_descriptions = load_clean_descriptions(descriptions_file, train_imgs)

def create_sequences(tokenizer, max_length, desc_list, feature):
    X1, X2, y = list(), list(), list()
    for desc in desc_list:
        seq = tokenizer.texts_to_sequences([desc])[0]
        for i in range(1, len(seq)):
            in_seq, out_seq = seq[:i], seq[i]
            #Post-padding to satisfy fast cuDNN acceleration
            in_seq = pad_sequences([in_seq], maxlen=max_length, padding='post')[0]
            out_seq = tf.keras.utils.to_categorical([out_seq], num_classes=vocab_size)[0]
            X1.append(feature)
            X2.append(in_seq)
            y.append(out_seq)
    return np.array(X1), np.array(X2), np.array(y)

def data_generator(descriptions, features, tokenizer, max_length):
    def generator():
        keys = list(descriptions.keys())
        while True:
            random.shuffle(keys)
            for key in keys:
                if key not in features: continue
                feature = features[key]
                input_image, input_sequence, output_word = create_sequences(tokenizer, max_length, descriptions[key], feature)
                for i in range(len(input_image)):
                    yield {'input_1': input_image[i], 'input_2': input_sequence[i]}, output_word[i]

    output_signature = (
        {
            'input_1': tf.TensorSpec(shape=(100, 2048), dtype=tf.float32),
            'input_2': tf.TensorSpec(shape=(max_length,), dtype=tf.int32)
        },
        tf.TensorSpec(shape=(vocab_size,), dtype=tf.float32)
    )
    return tf.data.Dataset.from_generator(generator, output_signature=output_signature).batch(64).prefetch(tf.data.AUTOTUNE)


class BahdanauAttention(Layer):
    def __init__(self, units, **kwargs):
        super(BahdanauAttention, self).__init__(**kwargs)
        self.W1 = Dense(units)
        self.W2 = Dense(units)
        self.V = Dense(1)

    def call(self, features, hidden):
        hidden_with_time_axis = tf.expand_dims(hidden, 1)

        # Shape score: (batch_size, 49, 1)
        score = self.V(tf.nn.tanh(self.W1(features) + self.W2(hidden_with_time_axis)))

        # Compute normalized attention weights
        attention_weights = tf.nn.softmax(score, axis=1)

        # Build context vector
        context_vector = attention_weights * features
        context_vector = tf.reduce_sum(context_vector, axis=1)

        return context_vector

def define_attention_model(vocab_size, max_length):
    # Inputs
    image_input = Input(shape=(100, 2048), name='input_1')
    seq_input = Input(shape=(max_length,), name='input_2')

    # Text Sequence Branch
    embedding = Embedding(vocab_size, 256, mask_zero=True)(seq_input)
    dropout_text = Dropout(0.4)(embedding)

    #512 LSTM units for enhanced context storage capacity
    lstm_layer = LSTM(512, return_state=True)
    lstm_out, state_h, state_c = lstm_layer(dropout_text)

    # Project raw image tokens into shared vector space
    image_features_projected = Dense(256, activation='relu')(image_input)

    # Calculate contextual visual attention maps targeting current LSTM output
    attention = BahdanauAttention(512)
    context_vector = attention(image_features_projected, lstm_out)

    # Concatenate visual context maps dynamically with textual hidden maps
    merged = concatenate([context_vector, lstm_out], axis=-1)

    decoder = Dense(256, activation='relu')(merged)
    output = Dense(vocab_size, activation='softmax')(decoder)

    model = Model(inputs=[image_input, seq_input], outputs=output)

    #Fine-tuned learning rate (0.0003) to enable stable convergence
    model.compile(loss='categorical_crossentropy', optimizer=tf.keras.optimizers.Adam(learning_rate=0.0003))
    return model

model = define_attention_model(vocab_size, max_length)

def get_steps_per_epoch(train_descriptions):
    total_sequences = 0
    for img_captions in train_descriptions.values():
        for caption in img_captions:
            total_sequences += len(caption.split()) - 1
    return max(1, total_sequences // 64)

steps = get_steps_per_epoch(train_descriptions)
dataset = data_generator(train_descriptions, train_features, tokenizer, max_length)

# Early stopping
callbacks_list = [
    EarlyStopping(monitor='loss', patience=3, restore_best_weights=True),
    ModelCheckpoint(os.path.join(models_dir, "best_attention_model.keras"), monitor='loss', save_best_only=True)
]

print("\nStarting Optimization Loop with Attention Architecture...")
model.fit(
    dataset,
    epochs=30,
    steps_per_epoch=steps,
    callbacks=callbacks_list,
    verbose=1
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Spatial features file found! Loading existing array maps...

Starting Optimization Loop with Attention Architecture...
Epoch 1/30
4787/4787 ━━━━━━━━━━━━━━━━━━━━ 523s 108ms/step - loss: 4.6606
Epoch 2/30
4787/4787 ━━━━━━━━━━━━━━━━━━━━ 517s 108ms/step - loss: 3.7303
Epoch 3/30
4787/4787 ━━━━━━━━━━━━━━━━━━━━ 511s 107ms/step - loss: 3.3402
Epoch 4/30
4787/4787 ━━━━━━━━━━━━━━━━━━━━ 508s 106ms/step - loss: 3.0646
Epoch 5/30
4787/4787 ━━━━━━━━━━━━━━━━━━━━ 508s 106ms/step - loss: 2.8406
Epoch 6/30
4787/4787 ━━━━━━━━━━━━━━━━━━━━ 509s 106ms/step - loss: 2.6441
Epoch 7/30
4787/4787 ━━━━━━━━━━━━━━━━━━━━ 506s 106ms/step - loss: 2.4724
Epoch 8/30
4787/4787 ━━━━━━━━━━━━━━━━━━━━ 504s 105ms/step - loss: 2.3143
Epoch 9/30
4787/4787 ━━━━━━━━━━━━━━━━━━━━ 505s 106ms/step - loss: 2.1726
Epoch 10/30
4787/4787 ━━━━━━━━━━━━━━━━━━━━ 501s 105ms/step - loss: 2.0440
Epoch 11/30
4787/4787

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.


KeyboardInterrupt

